# 16 — Hitta dolda/fjuniga mustascher i clean-mappen
Kör mustasch-detektorn på sin egen `clean`-mapp och sorterar efter `mustache_prob` fallande. Bilder med högst sannolikhet (även under 0.5) är de bästa kandidaterna för felmärkningar eller dolda fjuniga mustascher — samma teknik som `03_clean_mustache_dataset.ipynb` använde tidigare i projektet.

In [2]:
import os
import numpy as np
import tensorflow as tf
from PIL import Image

CLEAN_DIR = 'data/dataset_v3/clean'
MODEL_PATH = 'models/mustache_detector_3.keras'

model = tf.keras.models.load_model(MODEL_PATH)

files = [
    os.path.join(CLEAN_DIR, f) for f in os.listdir(CLEAN_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]
print(f'{len(files)} bilder i clean-mappen.')

5936 bilder i clean-mappen.


## Kör modellen på alla clean-bilder

In [3]:
results = []

for i, path in enumerate(files):
    try:
        img = Image.open(path).convert('RGB').resize((178, 178))
    except Exception:
        continue
    arr = np.expand_dims(np.array(img), axis=0).astype('float32')
    prob = float(model.predict(arr, verbose=0)[0][0])
    results.append((path, prob))

    if (i + 1) % 1000 == 0:
        print(f'{i + 1}/{len(files)} klara...')

results.sort(key=lambda x: x[1], reverse=True)
print('Klart! Topp 5 misstänkta:')
for path, prob in results[:5]:
    print(f'{os.path.basename(path)}: {prob:.3f}')

1000/5936 klara...
2000/5936 klara...
3000/5936 klara...
4000/5936 klara...
5000/5936 klara...
Klart! Topp 5 misstänkta:
155530.jpg: 0.999
177054.jpg: 0.997
152074.jpg: 0.995
080809.jpg: 0.994
165279.jpg: 0.981


## Visa de mest misstänkta kandidaterna
Höj `N` för att se fler. Granska visuellt och flytta de som genuint har en mustasch till `mustache`-mappen (eller en granskningsmapp för senare sortering).

In [ ]:
import matplotlib.pyplot as plt

N = 40
top_candidates = results[:N]

cols = 5
rows = (N + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 3.5 * rows))
axes = np.array(axes).flatten()

for ax, (path, prob) in zip(axes, top_candidates):
    img = Image.open(path)
    ax.imshow(img)
    ax.set_title(f'{os.path.basename(path)}\nprob={prob:.3f}', fontsize=8)
    ax.axis('off')

for ax in axes[len(top_candidates):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

## Flytta godkända fynd till granskningsmapp
Fyll i filnamnen (inte hela sökvägen) för bilder du visuellt bedömt har en genuin (om än svag) mustasch.

In [4]:
import shutil

REVIEW_DIR = 'data/clean_hidden_mustache_review'
os.makedirs(REVIEW_DIR, exist_ok=True)

TOP_N = 500
to_move = results[:TOP_N]

moved = 0
for path, prob in to_move:
    fname = os.path.basename(path)
    if os.path.exists(path):
        shutil.move(path, os.path.join(REVIEW_DIR, fname))
        moved += 1

print(f'{moved} bilder (topp {TOP_N} efter mustache_prob) flyttade till {REVIEW_DIR}.')
print('Gå igenom mappen manuellt — flytta de som genuint har en mustasch till dataset_v3/mustache,')
print('och flytta resten tillbaka till dataset_v3/clean.')

500 bilder (topp 500 efter mustache_prob) flyttade till data/clean_hidden_mustache_review.
Gå igenom mappen manuellt — flytta de som genuint har en mustasch till dataset_v3/mustache,
och flytta resten tillbaka till dataset_v3/clean.
